# Ingesta pipeline — end-to-end through the API

Exercises the whole T17 pipeline exactly as a real client would: upload a document
over HTTP, watch the SSE event stream, check the review queue, and record a human
decision. No shortcuts through the coordinator or the nodes directly — every call
below goes through `TestClient` and the same FastAPI routes a frontend would call.

Uses `TestContainer` (in-memory repos, stub mime detector, stub embeddings, `MockLlm`
for the SLM calls) so nothing here needs a real database, a real model file, or
network access — same reasoning as the test suite.

> **Kernel**: select the project's `.venv` kernel in the top-right picker.


## 1 — App setup: TestContainer wired into the real Container, JWT auth

In [1]:
from fastapi.testclient import TestClient

from classiflow.api.app import create_app
from classiflow.database.models import AllowedUser
from classiflow.injections.production import Container
from classiflow.injections.test import TestContainer
from classiflow.services.auth import encode_token
from classiflow.settings import Settings

Settings.JWT_SECRET_KEY = "playground-secret-key-not-for-prod-use-only-demo"

# Provide[Container.x] markers throughout the app point at the *production* Container
# class by identity, so we override it with a TestContainer instance rather than wire
# TestContainer directly -- same pattern tests/api/conftest.py uses.
test_container = TestContainer()
container = Container()
container.override(test_container)
container.wire(packages=["classiflow"])

_EMAIL = "leonardo.heis@gmail.com"
test_container.user_repo().seed(AllowedUser(email=_EMAIL, is_active=True, is_blocked=False))

client = TestClient(create_app())
auth_headers = {"Authorization": f"Bearer {encode_token(_EMAIL)}"}

print(f"logged in as {_EMAIL}")

c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
c:\Users\leona\source\repos\Trabajo-Integrador\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


logged in as leonardo.heis@gmail.com


## 2 — Real sample PDFs and the SLM mock

Two actual municipal documents from `playground/samples/`, not synthetic bytes — real
file size, real magic-byte MIME sniffing, real SHA-256. Node 3's legitimacy check
calls an LLM in production; here we swap it for `MockLlm` so the notebook doesn't
need a real model file. `set_legitimacy(...)` toggles what the mocked SLM decides,
which is what pushes a document toward `accepted` vs `review` further down.

> Text extraction is real now (T21: MarkItDown -> EasyOCR fallback — see
> `playground/stage1/text_extraction.ipynb` for that on its own). This notebook still
> uses `TestContainer`, which stubs it to a fixed sample Spanish text for speed and
> determinism (real OCR can take minutes per scanned document). So `passed`/`review`
> here comes entirely from `set_legitimacy(...)`, not from anything actually read out
> of the uploaded file.

In [2]:
from pathlib import Path

import classiflow
import classiflow.ingesta.nodes.node3_content_validation as node3_module
from classiflow.ingesta.llm_provider import MockLlm

_SAMPLES_DIR = Path(classiflow.__file__).parent / "playground" / "samples"
_ACCEPTED_PDF = (_SAMPLES_DIR / "convenio_2_2013.pdf").read_bytes()
_REVIEW_PDF = (_SAMPLES_DIR / "boletin_65_2005_doc_39153.pdf").read_bytes()

_SLM_LEGITIMATE = '{"is_legitimate": true, "confidence": 0.92, "reasoning": "official doc"}'
_SLM_NOT_LEGITIMATE = '{"is_legitimate": false, "confidence": 0.88, "reasoning": "looks like spam"}'


def set_legitimacy(*, is_legitimate: bool) -> None:
    response = _SLM_LEGITIMATE if is_legitimate else _SLM_NOT_LEGITIMATE
    node3_module.get_llm_langchain = lambda _path: MockLlm(response=response)


def upload(filename: str, file_bytes: bytes) -> dict[str, tuple[str, bytes, str]]:
    return {"file": (filename, file_bytes, "application/pdf")}


print(f"accepted-demo PDF: {len(_ACCEPTED_PDF):,} bytes")
print(f"review-demo PDF  : {len(_REVIEW_PDF):,} bytes")

accepted-demo PDF: 121,098 bytes
review-demo PDF  : 1,636,073 bytes


## 3 — Happy path: ingest a legitimate document

`POST /pipeline/ingest` returns `202` + a `job_id` immediately. `TestClient` runs
FastAPI's background tasks synchronously as part of the call, so by the time this
returns, the coordinator has already run node1 -> node2 -> node3 -> node4 to
completion -- in a real deployment this would happen after the response, which is
what `GET /{job_id}/events` (next section) is for: watching it happen live instead of
after the fact.

In [3]:
set_legitimacy(is_legitimate=True)

response = client.post(
    "/pipeline/ingest", files=upload("convenio_2_2013.pdf", _ACCEPTED_PDF), headers=auth_headers
)
print(f"status: {response.status_code}")
print(f"body  : {response.json()}")

accepted_job_id = response.json()["jobId"]

2026-08-11 23:36:04.561 | INFO     | classiflow.services.audit.service:record:37 - audit | job=98738f55-fb18-4538-97ad-10079f83cc05 node=node1_file_reception event=passed
2026-08-11 23:36:04.569 | INFO     | classiflow.services.audit.service:record:37 - audit | job=98738f55-fb18-4538-97ad-10079f83cc05 node=node2_format_validation event=passed
2026-08-11 23:36:04.952 | INFO     | classiflow.services.audit.service:record:37 - audit | job=98738f55-fb18-4538-97ad-10079f83cc05 node=node3_content_validation event=passed
2026-08-11 23:36:04.955 | INFO     | classiflow.services.audit.service:record:37 - audit | job=98738f55-fb18-4538-97ad-10079f83cc05 node=node4_duplicate_control event=passed


status: 202
body  : {'jobId': '98738f55-fb18-4538-97ad-10079f83cc05'}


## 4 — Watch the SSE event stream

Streams `node_update` events as they were emitted — `started` then `passed`/`failed`
per node — ending with a `pipeline` node event carrying `status: done`, at which
point the stream closes.

In [4]:
response = client.get(f"/pipeline/{accepted_job_id}/events", headers=auth_headers)
print(f"status: {response.status_code}\n")

for raw_block in response.text.split("event: node_update"):
    stripped = raw_block.strip()
    if stripped:
        print(stripped.removeprefix("data: "))

status: 200

{"job_id":"98738f55-fb18-4538-97ad-10079f83cc05","node":"node1_file_reception","status":"started","timestamp":"2026-08-12T02:36:04.561388Z","detail":{}}
{"job_id":"98738f55-fb18-4538-97ad-10079f83cc05","node":"node1_file_reception","status":"passed","timestamp":"2026-08-12T02:36:04.561969Z","detail":{}}
{"job_id":"98738f55-fb18-4538-97ad-10079f83cc05","node":"node2_format_validation","status":"started","timestamp":"2026-08-12T02:36:04.568501Z","detail":{}}
{"job_id":"98738f55-fb18-4538-97ad-10079f83cc05","node":"node2_format_validation","status":"passed","timestamp":"2026-08-12T02:36:04.569024Z","detail":{}}
{"job_id":"98738f55-fb18-4538-97ad-10079f83cc05","node":"node3_content_validation","status":"started","timestamp":"2026-08-12T02:36:04.572167Z","detail":{}}
{"job_id":"98738f55-fb18-4538-97ad-10079f83cc05","node":"node3_content_validation","status":"passed","timestamp":"2026-08-12T02:36:04.952167Z","detail":{}}
{"job_id":"98738f55-fb18-4538-97ad-10079f83cc05","node":"n

## 5 — Review queue is empty

The document was accepted, so it never shows up in `GET /pipeline/review-queue`.

In [7]:
queue = client.get("/pipeline/review-queue", headers=auth_headers).json()
job_ids_in_queue = [item["jobId"] for item in queue]

print(f"jobs currently in review: {len(queue)}")
assert accepted_job_id not in job_ids_in_queue
print("accepted job correctly absent from the review queue")

jobs currently in review: 1
accepted job correctly absent from the review queue


## 6 — A document the SLM flags for review

Same upload, but this time the mocked SLM says the content isn't legitimate. Node 3
sets `needs_agent_review=True`, the coordinator routes to `review` instead of
`accepted`/`rejected`, and the job lands in the review queue with its full
`document_steps` history attached.

In [6]:
set_legitimacy(is_legitimate=False)

response = client.post(
    "/pipeline/ingest",
    files=upload("boletin_65_2005_doc_39153.pdf", _REVIEW_PDF),
    headers=auth_headers,
)
review_job_id = response.json()["jobId"]
print(f"ingested job_id: {review_job_id}")

queue = client.get("/pipeline/review-queue", headers=auth_headers).json()
item = next(i for i in queue if i["jobId"] == review_job_id)

print(f"\nstatus          : {item['status']}")
print(f"filename        : {item['filename']}")
print(f"rejection_reason: {item['rejectionReason']}")
print("\ndocument_steps:")
for step in item["documentSteps"]:
    print(f"  [{step['stepOrder']}] {step['node']:<28} status={step['status']}")

2026-08-11 23:36:18.818 | INFO     | classiflow.services.audit.service:record:37 - audit | job=65c08fa7-7397-4a9a-a220-17d1c8f46015 node=node1_file_reception event=passed
2026-08-11 23:36:18.823 | INFO     | classiflow.services.audit.service:record:37 - audit | job=65c08fa7-7397-4a9a-a220-17d1c8f46015 node=node2_format_validation event=passed
2026-08-11 23:36:18.831 | INFO     | classiflow.services.audit.service:record:37 - audit | job=65c08fa7-7397-4a9a-a220-17d1c8f46015 node=node3_content_validation event=failed


ingested job_id: 65c08fa7-7397-4a9a-a220-17d1c8f46015

status          : review
filename        : boletin_65_2005_doc_39153.pdf
rejection_reason: SLM: looks like spam

document_steps:
  [1] node1_file_reception         status=passed
  [2] node2_format_validation      status=passed
  [3] node3_content_validation     status=failed


## 7 — Record a human decision

A reviewer accepts the flagged document anyway. `POST /pipeline/{job_id}/decision`
records who decided and why, updates the job's status, and the job then disappears
from the review queue.

In [8]:
response = client.post(
    f"/pipeline/{review_job_id}/decision",
    json={"decision": "accept", "notes": "Verified manually, looks legitimate"},
    headers=auth_headers,
)
print(f"status: {response.status_code}")

queue = client.get("/pipeline/review-queue", headers=auth_headers).json()
print(f"still in review queue: {review_job_id in [i['jobId'] for i in queue]}")

status: 200
still in review queue: False


## 8 — Guardrails: auth, unknown jobs, wrong state

Quick check that the error paths behave as designed:
- No token -> `401`
- Unknown `job_id` -> `404`
- Deciding on a job that isn't in `review` anymore -> `409`

In [9]:
no_auth = client.post("/pipeline/ingest", files=upload("x.pdf", _ACCEPTED_PDF))
print(f"no auth header       -> {no_auth.status_code}")

unknown = client.get("/pipeline/no-such-job/events", headers=auth_headers)
print(f"unknown job events    -> {unknown.status_code}")

already_decided = client.post(
    f"/pipeline/{review_job_id}/decision",
    json={"decision": "accept"},
    headers=auth_headers,
)
print(f"decide on non-review  -> {already_decided.status_code}")

no auth header       -> 401
unknown job events    -> 404
decide on non-review  -> 409
